Azure Service Principal reference notebook
*Co-authored with CoCo*

# What is a Tenant?

A **tenant** is an isolated, dedicated instance of an identity/account system that an organization gets when they sign up for a cloud platform. Think of it as your organization's **private boundary** within a shared cloud infrastructure.

- Tenants can be individuals, organizations, or departments within an organization.
- In **multi-tenant architectures**, multiple tenants share the same infrastructure, but each tenant's data is isolated from others.
- Not all platforms use the word "tenant" — some call it an "account" or "organization" — but the concept of isolation exists everywhere.

---

## Azure Entra ID Tenant

In Azure, a tenant is an **Entra ID directory** — the container for all identity objects (users, groups, apps, service principals) belonging to your organization.

| What it contains | Examples |
|-----------------|----------|
| Users | employees, admins |
| Groups | security groups, distribution lists |
| App Registrations | applications your org owns |
| Service Principals | app instances (yours + external apps you've consented to) |
| Policies | conditional access, MFA, network policies |

Every Azure subscription is associated with exactly **one** Entra ID tenant. The tenant is the identity boundary — it controls who can authenticate and what they can access.

---

## How Snowflake Interacts with Your Azure Tenant

Snowflake (the company) has its **own** Entra ID tenant. When you create a storage integration, Snowflake's app (registered in their tenant) gets "installed" into your tenant via the consent flow:

```
Snowflake's Entra ID Tenant              YOUR Entra ID Tenant
┌──────────────────────────┐             ┌──────────────────────────┐
│                          │             │                          │
│  App Registration        │             │  Service Principal       │
│  (Snowflake's app)       │── consent ──►  (instance of their app  │
│                          │   creates      now lives HERE)         │
│                          │             │                          │
└──────────────────────────┘             └──────────────────────────┘
```

Snowflake references your `AZURE_TENANT_ID` so it knows which token endpoint to authenticate against.

---

## Identities Do Not Cross Tenant or Platform Boundaries

An identity created in one tenant/platform **cannot be used directly** in another:

| Identity | Exists only in |
|----------|----------------|
| Azure Service Principal | Your specific Entra ID tenant |
| Snowflake Service User | Your Snowflake account |
| Databricks Service Principal | Your Databricks account |

To connect across boundaries, platforms use **federation** — trust relationships established via OAuth2, consent URLs, or managed identities.

---

## Summary

| Concept | In Azure context |
|---------|------------------|
| **Tenant** | Your Entra ID directory — the identity boundary for your org |
| **Tenant ID** | A GUID that uniquely identifies your directory |
| **Cross-platform access** | Works through consent/federation, not by sharing identities |
| **Why Snowflake needs your Tenant ID** | To authenticate against your tenant's token endpoint and get access to your storage |

# Tenants and Cross-Platform Identity

---

## What is a Tenant?

A **tenant** is an isolated, dedicated instance of an identity/account system that an organization gets when they sign up for a platform. Think of it as your organization's **private boundary** within a shared cloud platform.

---

## Tenant Concepts Across Platforms

| Platform | Has tenant concept? | What they call it |
|----------|:-------------------:|-------------------|
| **Azure** | Yes | **Entra ID Tenant** — a directory of users, apps, service principals |
| **AWS** | Sort of | **AWS Account** — serves similar isolation purpose, no "tenant" terminology |
| **Snowflake** | Sort of | **Snowflake Account** — isolated environment per customer |
| **Databricks** | Sort of | **Databricks Workspace / Account** |
| **Google Cloud** | Yes | **GCP Organization / Project** |

Azure is the platform where the "tenant" concept is most explicit — because Entra ID is fundamentally a multi-tenant identity provider.

---

## Identities Do Not Cross Platform Boundaries

A service principal / service identity is **specific to the platform** it was created in:

| Identity | Exists only in |
|----------|----------------|
| Azure Service Principal | Entra ID |
| AWS IAM Role | AWS IAM |
| Snowflake Service User | Snowflake |
| Databricks Service Principal | Databricks |
| GCP Service Account | Google Cloud IAM |

You **cannot** take an Azure service principal and use it directly in AWS or Snowflake. They are not interchangeable.

---

## How Cross-Platform Access Works: Federation

Platforms connect through **trust relationships** (federation), not by sharing identities:

```
Platform A                                   Platform B
┌─────────────────────┐                     ┌─────────────────────┐
│                     │                     │                     │
│  Identity exists    │── trust/federation──►  Identity exists    │
│  HERE only          │   (OAuth, SAML,     │  HERE only          │
│                     │    STS AssumeRole)   │                     │
└─────────────────────┘                     └─────────────────────┘
```

### Examples of federation in practice:

| Integration | How it works |
|-------------|---------------|
| **Snowflake → Azure Storage** | Snowflake creates a SP in its Entra ID tenant; you consent to install it in your tenant |
| **Snowflake → AWS S3** | Snowflake assumes an **IAM role** in your AWS account via STS AssumeRole |
| **Databricks → Azure Storage** | Databricks uses an Entra ID SP (linked or created separately) |
| **Any platform → GCP Storage** | Uses GCP service account keys or Workload Identity Federation |

Each platform's identity stays within that platform's boundary. The connection happens via **federation protocols** — the identity itself never moves.

---

## Key Takeaway

- **Tenant** = your org's isolated boundary on a given platform
- **Each platform has its own identity system** — they don't share service principals
- **Cross-platform access** works through trust/federation, not by moving identities across platforms

# Service Principal in Azure

A **service principal** is a security identity used by applications, services, or automation tools to access resources without user interaction. It's essentially a service account with specific permissions, enabling secure, non-interactive authentication.

In Azure, a service principal is a **Microsoft Entra ID** (formerly Azure AD) object representing an application within a tenant. It defines what the application can do, which resources it can access, and who can use it. Service principals are created automatically when you register an application in Microsoft Entra ID and are ideal for CI/CD pipelines, cross-cloud deployments, and automation scripts.

---

## Key Azure Characteristics

- Created via **application registration** in Microsoft Entra ID.
- Supports **OAuth 2.0** and **OpenID Connect** authentication.
- Requires credentials (certificate or client secret).
- Can be multi-tenant for cross-organization access.

---

## App Registration vs Service Principal

These are two distinct but related objects in Entra ID:

| Object | What it is | Where you see it |
|--------|-----------|------------------|
| **App Registration** | The global application definition (template) — defines client ID, credentials, permissions | Entra ID → App registrations |
| **Service Principal** | The local instance of that app within a specific tenant | Entra ID → Enterprise applications |

A service principal **cannot exist without** an underlying app registration in Azure. The app registration defines the identity, and the service principal is the "instantiation" of that app within a tenant.

### Key rules:
- **One app registration** can have **multiple service principals** — one in each tenant that uses the app (multi-tenant pattern)
- Within the **same tenant**, the relationship is 1:1 (one app registration = one service principal)
- If you need multiple identities within the same tenant, create multiple app registrations

### Analogy:
- **App Registration** = an app on the App Store (developer owns it)
- **Service Principal** = the installed instance on your phone (you consented to install it)
- You can grant it permissions (camera, storage) but you don't own the source code

---

## Creating a Service Principal via CLI

```bash
az ad sp create-for-rbac \
  --name "myApp" \
  --role Contributor \
  --scopes /subscriptions/{subscription-id}/resourceGroups/{resource-group}
```

### What this command does behind the scenes:

1. Creates an **App Registration** named "myApp" (generates an `appId` / client ID)
2. Creates a **Service Principal** tied to that app registration in your tenant
3. Generates a **client secret** (password) for authentication

### Output:

```json
{
  "appId": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx",    // App Registration's client ID
  "displayName": "myApp",
  "password": "xxxxxxxx",                              // Client secret
  "tenant": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
}
```

> **Note:** `az ad sp create-for-rbac` creates **both** the app registration and the service principal — not just the service principal.

---

## Best Practices

- **Least privilege:** Assign only required permissions.
- **Credential rotation:** Use certificates or managed identities; rotate secrets regularly.
- **Automation:** Use service principals for scripts instead of personal credentials.
- **Audit:** Monitor usage via logs in Azure Monitor.

---

## Summary

Azure service principals are application identities in Entra ID enabling secure, automated access without tying to a human user.

**Resource:** [Azure Service Principals Explained](https://cloudwebschool.com/docs/azure/iam-and-security/azure-service-principals-explained/)

# Multiple Service Principals from One App Registration

A single app registration can have **multiple service principals** — one in each tenant where the app is used. This is the **multi-tenant app** pattern.

---

## The Relationship

```
App Registration (Home Tenant A)          ← Only ONE exists (the definition)
  ├── Service Principal in Tenant A       ← home tenant (auto-created)
  ├── Service Principal in Tenant B       ← created when Tenant B admin consents
  └── Service Principal in Tenant C       ← created when Tenant C admin consents
```

Each tenant gets its own service principal when an admin grants consent (via a consent URL or admin consent flow).

---

## Each Service Principal Is Independent

Even though they all come from the same app registration, each tenant's service principal can have different:

| Property | Controlled per tenant |
|----------|----------------------|
| **Role assignments (RBAC)** | Each tenant assigns its own roles |
| **Permission grants** | Each tenant controls what it allows |
| **Conditional access policies** | Each tenant applies its own policies |

---

## Real-World Example: Snowflake Storage Integration

- Snowflake has **one app registration** in their Entra ID tenant
- Every Snowflake customer who creates a storage integration and visits the consent URL gets a **separate service principal** in their own tenant
- Each customer assigns IAM roles to their own instance independently

```
Snowflake's Tenant (App Registration)
  ├── SP in Customer A's Tenant  → Reader on Container X
  ├── SP in Customer B's Tenant  → Contributor on Container Y
  └── SP in Customer C's Tenant  → Reader on Container Z
```

---

## Rule: 1:1 Within the Same Tenant

Within a **single tenant**, the relationship is strictly **one app registration = one service principal**.

| Scope | Multiple SPs possible? |
|-------|------------------------|
| Across different tenants | **Yes** — one SP per tenant |
| Within the same tenant | **No** — always 1:1 |

If you need multiple identities within the same tenant, create **multiple app registrations**.

# Multi-Tenant App Access — How a 3rd Party Accesses Your Resources

When you register an app in your Azure tenant and make it multi-tenant, other tenants can use it. But you don't "pass" the service principal to them — **Azure creates a separate service principal in each tenant upon consent**.

---

## Making Your App Available to Another Tenant

### What you have:

```
YOUR Tenant (Tenant A)
┌─────────────────────────────────┐
│  App Registration: "MyDataApp"  │
│  - appId: abc-123               │
│  - Multi-tenant: YES            │  ← set in app settings
│  - Service Principal (your own) │
└─────────────────────────────────┘
```

### How another tenant gets access:

The other tenant's admin visits a consent URL:

```
https://login.microsoftonline.com/{their-tenant-id}/adminconsent?client_id=abc-123
```

### After consent:

```
YOUR Tenant (Tenant A)                    THEIR Tenant (Tenant B)
┌───────────────────────────┐            ┌───────────────────────────┐
│  App Registration         │            │  Service Principal        │
│  (definition stays here)  │            │  (auto-created by Azure   │
│                           │            │   when admin consented)   │
│  Service Principal        │            │                           │
│  (your local instance)    │            │  They grant it IAM roles  │
└───────────────────────────┘            └───────────────────────────┘
```

> **Important:** Each tenant gets its own **separate** service principal with a **different object ID**. The `appId` (client ID) is the same across all tenants, but the service principal object ID is unique per tenant.

---

## How the 3rd Party Actually Accesses Your Resources

### Step 1: Authentication — the 3rd party proves its identity

The 3rd party app (running on their servers) calls **your tenant's** token endpoint:

```http
POST https://login.microsoftonline.com/{YOUR-tenant-id}/oauth2/v2.0/token

client_id=abc-123                    ← the app's ID (same across tenants)
client_secret=their-secret           ← credential only the app owner holds
scope=https://storage.azure.com/.default
grant_type=client_credentials
```

They call **your** tenant's endpoint because the resource (storage) lives in **your** tenant.

---

### Step 2: Azure validates the request

Azure checks:

| Check | Result |
|-------|--------|
| Does this `client_id` have a service principal in this tenant? | Yes (you consented) |
| Are the credentials valid? | Yes |
| What permissions does this SP have? | Whatever IAM roles you assigned |

Azure issues an **access token** scoped to your tenant.

---

### Step 3: The 3rd party uses the token to access your resources

```http
GET https://myaccount.blob.core.windows.net/container/file.csv
Authorization: Bearer <access-token-from-step-2>
```

Azure Storage checks:

| Check | Result |
|-------|--------|
| Is this token valid? | Yes |
| Does the SP have "Storage Blob Data Reader" on this container? | Yes (you assigned it) |
| **Access granted** | |

---

## The Complete Picture

```
3rd Party's Server                  YOUR Azure Tenant
┌────────────────────┐             ┌────────────────────────────────────┐
│                    │             │                                    │
│  Holds:            │  1. Auth    │  Entra ID                          │
│  - client_id       │────────────►  - Validates credentials            │
│  - client_secret   │             │  - Checks SP exists in YOUR tenant │
│                    │◄────────────│  - Issues access token             │
│                    │  2. Token   │                                    │
│                    │             │  Azure Storage                     │
│                    │  3. Access  │  - Checks token + IAM roles        │
│                    │────────────►  - Grants/denies access             │
│                    │             │                                    │
└────────────────────┘             └────────────────────────────────────┘
```

---

## Service Principal IDs — Same or Different?

| Property | Same across tenants? | Explanation |
|----------|:--------------------:|-------------|
| `appId` (client ID) | **Yes** | Identifies the app globally |
| Service Principal **Object ID** | **No** | Each tenant has its own unique SP object |
| Credentials (secret/cert) | N/A | Only the app owner holds these |

So: the **app identity** is the same, but each tenant gets its own **independent service principal instance**.

---

## Your Control as the Resource Owner

| Action | How |
|--------|-----|
| **Grant access** | Assign IAM roles to the SP in your tenant |
| **Limit access** | Assign narrow roles (Reader vs Contributor) |
| **Revoke access** | Remove IAM roles, or delete the Enterprise Application from your tenant |
| **Audit access** | Check Azure Monitor / sign-in logs for the SP |

You never see or manage their credentials — you only control what the SP can do in your tenant.

---

## This Is Exactly What Snowflake Does

- Snowflake's servers hold the `client_id` + `client_secret` for their app
- After you consent, Snowflake authenticates against **your** tenant's token endpoint
- They receive a token and use it to read from your Azure Storage
- You control access by assigning (or removing) the **Storage Blob Data Reader** role

# How Snowflake Storage Integration Works Internally

Snowflake (the company) has its **own Azure Entra ID tenant** — just like your organization has one. Within their tenant, Snowflake has registered a **multi-tenant application** (app registration).

This section explains the complete internal flow of how Snowflake creates a service principal in your Azure tenant and uses it to access your storage.

---

## Step-by-Step Flow

### Step 1: You create the storage integration in Snowflake

```sql
CREATE STORAGE INTEGRATION my_azure_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'AZURE'
  AZURE_TENANT_ID = '<your-tenant-id>'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('azure://myaccount.blob.core.windows.net/mycontainer/');
```

You only provide your **Azure Tenant ID** and storage paths. No app registration or service principal is created by you.

### Step 2: Snowflake responds with a consent URL

Run `DESCRIBE INTEGRATION` to retrieve it:

```sql
DESCRIBE INTEGRATION my_azure_int;
```

This returns:
- `AZURE_CONSENT_URL` — a standard Azure OAuth2 admin consent URL
- `AZURE_MULTI_TENANT_APP_NAME` — name of Snowflake's registered app

The consent URL looks like:
```
https://login.microsoftonline.com/<your-tenant-id>/adminconsent?client_id=<snowflake-app-id>
```

### Step 3: Azure admin visits the consent URL and clicks "Accept"

When you accept, **Azure automatically creates a service principal** (Enterprise Application) in your tenant that references Snowflake's app registration:

```
Snowflake's Entra ID Tenant                YOUR Entra ID Tenant
┌────────────────────────────────┐         ┌────────────────────────────────┐
│                                │         │                                │
│  App Registration              │         │  Service Principal             │
│  ┌──────────────────────────┐  │         │  ┌──────────────────────────┐  │
│  │ appId: abc-123           │  │ consent │  │ References appId: abc-123│  │
│  │ Name: SnowflakeProd      │──┼────────►│  │ Visible under:           │  │
│  │ Multi-tenant: YES        │  │ creates │  │ "Enterprise Applications"│  │
│  │ Credentials: held by SF  │  │         │  │ NO credentials here      │  │
│  └──────────────────────────┘  │         │  └──────────────────────────┘  │
│                                │         │                                │
│  + Service Principal           │         │                                │
│  (Snowflake's own instance)    │         │                                │
└────────────────────────────────┘         └────────────────────────────────┘
```

### Step 4: Assign IAM roles to the service principal

In Azure Portal → your Storage Account → Access Control (IAM):

- Grant the newly-appeared service principal the **"Storage Blob Data Reader"** role (or "Storage Blob Data Contributor" if Snowflake needs write access)
- Scope it to the specific container or storage account

### Step 5: Snowflake accesses your storage (behind the scenes)

When Snowflake needs to read your data, its servers do the following:

```
1. Snowflake's server calls Azure Entra ID:
   POST https://login.microsoftonline.com/<YOUR-tenant-id>/oauth2/v2.0/token
   ├── client_id = <snowflake-app-id>
   ├── client_secret = <held by Snowflake — you never see this>
   ├── scope = https://storage.azure.com/.default
   └── grant_type = client_credentials

2. Azure validates:
   ├── Does this client_id have a service principal in this tenant? → YES (you consented)
   ├── Are credentials valid? → YES (Snowflake holds them)
   └── Issues an access token scoped to your tenant

3. Snowflake uses the token to access your storage:
   GET https://myaccount.blob.core.windows.net/mycontainer/data.csv
   Authorization: Bearer <access-token>

4. Azure Storage checks:
   ├── Is token valid? → YES
   ├── Does the service principal have "Storage Blob Data Reader"? → YES
   └── Access granted ✓
```

---

## Why You See It Under "Enterprise Applications" and NOT "App Registrations"

| Azure Portal Section | What it shows | Who owns it |
|---------------------|---------------|-------------|
| **App Registrations** | Apps **your** tenant created (the definition lives with you) | You |
| **Enterprise Applications** | Service principals in your tenant — including those from **external** apps you've consented to | The app owner (Snowflake in this case) |

Since the app registration belongs to **Snowflake's tenant**, you only see the service principal (Enterprise Application) in yours. You never see it under App Registrations because you didn't create it.

---

## Your Control Points

| What you control | How |
|-----------------|-----|
| **Grant access** | Assign IAM roles (Storage Blob Data Reader/Contributor) to the service principal |
| **Scope access** | Choose specific containers, storage accounts, or resource groups |
| **Revoke access** | Remove the IAM role assignment, or delete the Enterprise Application entirely |
| **Audit access** | Azure Monitor logs show every token request and storage access |

---

## What You Do NOT Control

- The **app registration** (Snowflake owns it)
- The **client secret / credentials** (Snowflake holds them — you never see or manage them)
- The **token refresh** (Snowflake handles this automatically)

---

## Summary

```
YOU do this:                              SNOWFLAKE does this:
─────────────────                         ─────────────────────
1. CREATE STORAGE INTEGRATION             → Provisions app for your tenant
2. DESCRIBE INTEGRATION                   → Returns consent URL
3. Visit consent URL → Accept             → Azure creates SP in your tenant
4. Assign IAM role to SP                  → Now has token-based access
5. Use external stage in queries          → Authenticates & reads your storage
```

The entire mechanism relies on **OAuth2 client_credentials flow** + **Azure IAM RBAC** — Snowflake authenticates as itself (via its app registration), and Azure authorizes access based on the roles you assigned to the service principal in your tenant.

# How Snowflake Storage Integration Works Internally with Azure

Snowflake (the company) has its **own Azure Entra ID tenant** — just like your organization has one. Within their tenant, Snowflake has registered a **multi-tenant application** (app registration).

---

## The Internal Flow

### Step 1: You create the storage integration

You provide your Azure tenant ID and storage paths:

```sql
CREATE STORAGE INTEGRATION my_azure_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'AZURE'
  AZURE_TENANT_ID = '<your-tenant-id>'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('azure://myaccount.blob.core.windows.net/mycontainer/');
```

---

### Step 2: Snowflake gives you a consent URL

Run `DESCRIBE INTEGRATION my_azure_int;` to get:
- `AZURE_CONSENT_URL` — a standard Azure OAuth2 admin consent link pointing to Snowflake's app registration
- `AZURE_MULTI_TENANT_APP_NAME` — the name of Snowflake's registered app

---

### Step 3: You (Azure admin) visit the consent URL and click "Accept"

When you accept, Azure performs the following:

```
Snowflake's Entra ID Tenant              YOUR Entra ID Tenant
┌──────────────────────────┐             ┌──────────────────────────┐
│                          │             │                          │
│  App Registration        │             │  Service Principal       │
│  (the original app       │── consent ──►  (an instance of         │
│   definition lives HERE) │   creates      Snowflake's app now     │
│                          │                lives HERE)             │
│  + Service Principal     │             │                          │
│  (Snowflake's own copy)  │             │                          │
└──────────────────────────┘             └──────────────────────────┘
```

**What consent actually does:** It creates a **service principal** (Enterprise Application entry) in your tenant that references Snowflake's app registration.

---

### Step 4: You assign IAM roles

Grant the newly-appeared service principal access to your storage:

**Azure Portal → Storage Account → Access Control (IAM) → Add Role Assignment**

Assign **Storage Blob Data Reader** (or Contributor) to the Snowflake service principal.

---

### Step 5: Snowflake accesses your storage

Snowflake's servers authenticate against **your tenant's** token endpoint using their app credentials, receive a token, and use it to read/write to your Azure Storage.

---

## Why It Appears Under Enterprise Applications, Not App Registrations

| Azure Portal Section | What it shows |
|---------------------|---------------|
| **App Registrations** | Apps **your tenant owns** (the definition lives with you) |
| **Enterprise Applications** | Service principals in your tenant — including those from **external** apps you've consented to |

Since the app registration belongs to **Snowflake's tenant**, you only see the service principal (Enterprise Application) in yours — never the app registration.

---

## Analogy: Mobile App Installation

| Concept | Real-world equivalent |
|---------|----------------------|
| **App Registration** | The app on the App Store (developer owns it) |
| **Service Principal** | The installed instance on your phone (you consented to install it) |
| **IAM Role Assignment** | Permissions you grant it (camera, storage, contacts) |
| **Revoking access** | Uninstalling the app from your phone |

You can grant or revoke permissions, but you don't own the source code — the developer (Snowflake) does.

# Snowflake Azure Storage Integration — Where Does the Service Principal Come From?

When creating a storage integration in Snowflake for Azure, you only provide `tenant_id` and `storage_paths` in SQL. You never create an app registration yourself — **Snowflake creates and owns the service principal for you** using a multi-tenant app registration pattern.

---

## The Complete Flow

### Step 1: Create the Storage Integration

```sql
CREATE STORAGE INTEGRATION my_azure_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'AZURE'
  AZURE_TENANT_ID = '<your-tenant-id>'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('azure://myaccount.blob.core.windows.net/mycontainer/');
```

---

### Step 2: Snowflake Provisions a Service Principal (in Snowflake's Tenant)

Run `DESCRIBE INTEGRATION` to retrieve the details:

```sql
DESCRIBE INTEGRATION my_azure_int;
```

**Response includes:**

| Property | Description |
|----------|-------------|
| `AZURE_CONSENT_URL` | URL you visit to grant consent |
| `AZURE_MULTI_TENANT_APP_NAME` | Name of Snowflake's app registration |

This service principal is **created and owned by Snowflake** in Snowflake's own Entra ID tenant. It's a multi-tenant enterprise application that Snowflake registers on your behalf.

---

### Step 3: Grant Consent via the Consent URL

An Azure AD admin visits the `AZURE_CONSENT_URL` and clicks **Accept**.

**What happens behind the scenes:**

```
Snowflake's Entra ID Tenant              YOUR Entra ID Tenant
┌──────────────────────────┐             ┌──────────────────────────┐
│                          │             │                          │
│  App Registration        │             │  Service Principal       │
│  (owned by Snowflake)    │── consent ──►  (auto-created in your   │
│                          │   creates      tenant upon consent)    │
│  + Service Principal     │             │                          │
│  (Snowflake's own copy)  │             │  Visible under:          │
│                          │             │  Enterprise Applications │
└──────────────────────────┘             └──────────────────────────┘
```

Consent authorizes Snowflake's service principal to access resources in **your** tenant. It creates an "Enterprise Application" entry representing Snowflake's app.

---

### Step 4: Assign IAM Roles to the Service Principal

After consent, go to:

**Azure Portal → Storage Account → Access Control (IAM) → Add Role Assignment**

Grant the Snowflake service principal (now visible in your tenant) one of:
- **Storage Blob Data Reader** — for read-only access
- **Storage Blob Data Contributor** — for read/write access

---

## Why You Don't Create the App Registration

Snowflake uses a **multi-tenant app registration pattern**:

- Snowflake maintains the app registration in **its own** Entra ID tenant
- The consent URL is the standard **OAuth2 admin consent flow** that "installs" that app into your tenant
- This is the same pattern used by any SaaS product that needs access to your Azure resources (Power BI, Databricks, etc.)

---

## Summary of the Sequence

```
1. You run CREATE STORAGE INTEGRATION (provide tenant_id + paths)
         │
         ▼
2. Snowflake provisions a service principal in its own tenant
         │
         ▼
3. You visit consent URL → Azure creates a service principal in YOUR tenant
         │
         ▼
4. You assign IAM role (Storage Blob Data Reader) to that service principal
         │
         ▼
5. Snowflake can now access your Azure Storage ✓
```

You never need to create your own app registration in Azure for this path — Snowflake handles the identity lifecycle entirely.

# Can You Use Your Own Service Principal for Snowflake Storage Integration?

**No.** Snowflake does not support supplying your own app registration or custom service principal for storage integrations. The integration flow is fixed — Snowflake creates and manages the service principal entirely.

---

## Why You Can't Bring Your Own

The storage integration is designed as a **managed identity pattern**:

| Step | Who does it | What happens |
|------|-------------|--------------|
| 1 | You | Run `CREATE STORAGE INTEGRATION` (provide tenant_id + paths) |
| 2 | Snowflake | Automatically provisions a service principal in Snowflake's Entra ID tenant |
| 3 | You | Visit consent URL → "installs" Snowflake's app into your tenant |
| 4 | You | Assign IAM roles to the Snowflake-managed service principal |

You have **no option** to:
- Supply your own `client_id` / `client_secret`
- Point the integration to your own app registration
- Manage the service principal lifecycle yourself

This is intentional — Snowflake handles client secrets, certificate rotation, and token refresh so you don't have to.

---

## Alternative: SAS Tokens (Bring Your Own Credentials)

If you need full control over authentication (e.g., using your own Azure identity), skip storage integrations and use **SAS tokens** directly on external stages:

```sql
CREATE STAGE my_stage
  URL = 'azure://myaccount.blob.core.windows.net/container/'
  CREDENTIALS = (AZURE_SAS_TOKEN = '?sv=2021-06-08&ss=b&srt=sco&sp=rl...');
```

---

## Trade-offs: Storage Integration vs SAS Token

| Aspect | Storage Integration | SAS Token on Stage |
|--------|--------------------|-----------------------|
| **Who manages the identity?** | Snowflake (fully managed) | You (self-managed) |
| **Credential rotation** | Automatic (Snowflake handles it) | Manual (tokens expire, you must regenerate) |
| **Secrets in Snowflake** | None (no secrets stored) | SAS token stored in stage definition |
| **Setup complexity** | Simple (consent URL + IAM role) | More involved (generate token, manage expiry) |
| **Your control** | Limited (consent + IAM only) | Full control |
| **Security risk** | Lower (no secrets to leak) | Higher (token can be exposed) |

---

## When to Use Which

| Scenario | Recommended approach |
|----------|---------------------|
| Standard data loading / unloading | Storage Integration |
| Organization requires self-managed credentials | SAS Token |
| Temporary / short-lived access | SAS Token |
| Long-term production pipelines | Storage Integration |
| Compliance requires you to own all identities | SAS Token |

# Why Use Integration Objects When You Can Call Azure APIs Directly?

Yes, you **can** call Azure APIs directly from Snowflake (via Python in notebooks, Snowpark stored procedures, or external functions). But this does **not** replace integration objects — they serve completely different purposes.

---

## The Core Reason

Snowflake's built-in data services **require** an integration object. It's a **hard requirement**, not a preference.

```sql
-- This WILL NOT work without a storage integration:
COPY INTO my_table FROM @my_external_stage;

-- External stages REQUIRE a storage integration:
CREATE STAGE my_stage
  URL = 'azure://myaccount.blob.core.windows.net/container/'
  STORAGE_INTEGRATION = my_int;   -- MANDATORY

-- Snowpipe REQUIRES a notification integration:
CREATE PIPE my_pipe AUTO_INGEST = TRUE AS
  COPY INTO my_table FROM @my_stage;
```

If you want to use COPY INTO, Snowpipe, external stages, or external tables — you **must** create a storage/notification integration. There is no alternative.

---

## Two Separate Worlds

| Approach | What it does | When to use |
|----------|-------------|-------------|
| **Integration object** | Enables Snowflake's built-in services (COPY INTO, Snowpipe, external stages, external tables) to access Azure storage | Loading/unloading data at scale using Snowflake's engine |
| **Python / Azure APIs** | Your code calls Azure APIs directly using credentials you manage | Custom logic, transformations, calling non-storage Azure services (Cosmos DB, Service Bus, etc.) |

These are **not interchangeable**. They serve different purposes.

---

## What Happens If You Only Use Python (No Integration)

```python
# This works — you can read blobs via Python
from azure.storage.blob import BlobServiceClient

client = BlobServiceClient(account_url="https://myaccount.blob.core.windows.net", credential=sas_token)
blob = client.get_container_client("data").download_blob("file.csv").readall()
```

But now you **cannot** use any of these Snowflake features:

| Feature | Status without integration |
|---------|---------------------------|
| `COPY INTO table FROM @stage` | Will not work |
| `Snowpipe` (auto-ingest) | Will not work |
| `External Stage` | Will not work |
| `External Tables` | Will not work |
| `SELECT FROM @stage` | Will not work |
| `UNLOAD` to Azure | Will not work |

All of these services are **engine-level** — they run inside Snowflake's compute infrastructure and need a registered integration object to know how to authenticate.

---

## Why Can't Snowflake Just Accept Custom Credentials at Runtime?

Snowflake's architecture separates **who defines access** from **who uses it**:

```sql
-- Admin creates the integration (defines access)
CREATE STORAGE INTEGRATION my_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'AZURE'
  AZURE_TENANT_ID = '<tenant-id>'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('azure://myaccount.blob.core.windows.net/approved-container/');

-- Admin grants usage to a role
GRANT USAGE ON INTEGRATION my_int TO ROLE data_engineer;

-- Data engineer creates a stage (uses access)
CREATE STAGE my_stage
  URL = 'azure://myaccount.blob.core.windows.net/approved-container/sales/'
  STORAGE_INTEGRATION = my_int;

-- Data engineer loads data
COPY INTO sales_table FROM @my_stage FILE_FORMAT = (TYPE = CSV);
```

The integration enforces:
- **Security boundary:** `STORAGE_ALLOWED_LOCATIONS` prevents anyone from pointing a stage at unauthorized paths
- **Credential isolation:** Data engineers never see the SP credentials
- **Centralized management:** One integration object, many stages/pipes reference it

---

## Summary

| Question | Answer |
|----------|--------|
| Can you call Azure APIs from Snowflake? | **Yes** — via Python, Snowpark, external functions |
| Does that replace storage/notification integrations? | **No** — Snowflake's built-in services won't work without them |
| Why not? | COPY INTO, Snowpipe, external stages are engine-level services that **require** a registered integration object — they don't accept custom credentials |
| Are the two approaches competing? | **No** — they serve completely different use cases |

# Snowflake Service User — The Native "Service Principal" for API Access

Snowflake's equivalent of a service principal is a **Service User** (`TYPE = SERVICE`). It is a non-interactive identity designed for machine-to-machine communication and API access.

---

## Service User Characteristics

| Property | Behavior |
|----------|----------|
| Interactive login (UI) | Not allowed |
| Password-based auth | Not allowed |
| Key-pair authentication | Supported |
| Programmatic Access Tokens | Supported |
| Designed for | Automation, APIs, CI/CD, ETL pipelines |

---

## Creating a Service User

```sql
CREATE USER my_service_user
  TYPE = SERVICE
  COMMENT = 'Service account for API access';
```

---

## Authentication Option 1: Key-Pair Authentication

### Step 1: Generate a key pair locally

```bash
# Generate private key (PKCS8 format, no passphrase)
openssl genrsa 2048 | openssl pkcs8 -topk8 -inform PEM -out rsa_key.p8 -nocrypt

# Generate corresponding public key
openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub
```

### Step 2: Assign the public key to the service user

```sql
-- Paste the public key content (without BEGIN/END headers)
ALTER USER my_service_user SET RSA_PUBLIC_KEY = 'MIIBIjANBgkqh...';
```

### Step 3: Grant a role

```sql
GRANT ROLE my_api_role TO USER my_service_user;
ALTER USER my_service_user SET DEFAULT_ROLE = my_api_role;
```

The service user now authenticates using the **private key** — no password, no interactive login.

---

## Authentication Option 2: Programmatic Access Tokens (PAT)

For calling Snowflake REST APIs or SQL API, you can generate a programmatic access token:

```sql
SELECT SYSTEM$GENERATE_PROGRAMMATIC_ACCESS_TOKEN('my_service_user');
```

### Requirements for PAT:

- The service user **must** be subject to a **network policy** (with at least one network rule)
- The caller must own the target user or have appropriate privileges

---

## Snowflake User Types

| TYPE | Purpose | Allowed Auth Methods |
|------|---------|---------------------|
| `PERSON` | Human interactive users | Password, MFA, SSO, key-pair, PAT |
| `SERVICE` | Programmatic / API access | Key-pair, PAT only |
| `LEGACY_SERVICE` | Older service accounts (pre-existing) | Same as SERVICE (grandfathered) |

---

## Key Takeaway

To create a "service principal" in Snowflake for API access:

```sql
CREATE USER my_service_user TYPE = SERVICE;
```

Then authenticate it using either **key-pair** (for drivers/connectors) or **programmatic access tokens** (for REST APIs). No external identity provider is required — this is a purely Snowflake-native identity.

# Service Principal in Snowflake

Snowflake does **not** have a first-class "Service Principal" object like Databricks. Instead, it uses **service users** with various authentication mechanisms for machine-to-machine access.

---

## Snowflake's Approach: Service Users

Instead of a dedicated "service principal" construct, Snowflake provides:

| Mechanism | Purpose |
|-----------|---------|
| **Service User** (`TYPE = SERVICE`) | Non-interactive identity for automation |
| **Key-pair authentication** | Authenticate using RSA key pairs |
| **OAuth** | Token-based authentication |
| **Workload Identity Federation (WIF)** | Authenticate using cloud-native identities (Entra ID, AWS IAM, GCP SA) |

---

## Creating a Service User in Snowflake

### Option 1: Key-Pair Authentication (simplest)

```sql
-- Step 1: Create the service user
CREATE USER my_service_user
  TYPE = SERVICE
  COMMENT = 'Automation account for ETL pipelines';

-- Step 2: Assign the public key
ALTER USER my_service_user SET RSA_PUBLIC_KEY = 'MIIBIjANBgkqh...';

-- Step 3: Grant a role
GRANT ROLE etl_role TO USER my_service_user;
ALTER USER my_service_user SET DEFAULT_ROLE = etl_role;
```

The service user authenticates using the private key — no password, no interactive login.

---

### Option 2: Workload Identity Federation (cloud-native)

WIF lets a cloud workload (Azure VM, AWS Lambda, GCP Cloud Function) authenticate to Snowflake using its **native cloud identity** — no secrets to manage.

```sql
-- Azure Entra ID federation
CREATE USER wif_azure_user
  TYPE = SERVICE
  WORKLOAD_IDENTITY = (
    TYPE = AZURE
    AUDIENCE = 'https://login.microsoftonline.com/{tenant-id}'
    ISSUER = 'https://login.microsoftonline.com/{tenant-id}/v2.0'
    SUBJECT = '{app-id}'           -- The Entra ID app's client ID
  );

GRANT ROLE data_pipeline_role TO USER wif_azure_user;
```

```sql
-- AWS IAM federation
CREATE USER wif_aws_user
  TYPE = SERVICE
  WORKLOAD_IDENTITY = (
    TYPE = AWS
    AUDIENCE = 'urn:snowflake:account:{account-identifier}'
    ISSUER = 'https://sts.amazonaws.com'
    SUBJECT = '{aws-iam-role-arn}'
  );
```

The workload authenticates using its cloud identity token — Snowflake trusts it via federation.

---

## Databricks vs Snowflake Comparison

| Aspect | Databricks | Snowflake |
|--------|-----------|----------|
| **Native SP object?** | Yes (Databricks Service Principal) | No (uses Service Users) |
| **Created via** | Account Console, SCIM API, Terraform | `CREATE USER ... TYPE = SERVICE` |
| **Requires App Registration?** | No | No (unless using Entra ID WIF) |
| **Authentication methods** | Databricks OAuth, PATs | Key-pair, OAuth, WIF, Programmatic Access Tokens |
| **Can federate to cloud identity?** | Yes (link to Entra ID SP) | Yes (WIF to Entra ID, AWS IAM, GCP SA) |

---

## When Does an Entra ID App Registration Come Into Picture?

It depends on the authentication method you choose:

| Snowflake Auth Method | App Registration needed? |
|-----------------------|-------------------------|
| Key-pair authentication | No |
| Programmatic Access Token | No |
| OAuth (Snowflake-managed) | No |
| WIF with **Azure Entra ID** | **Yes** — you need an App Registration in Entra ID whose identity Snowflake trusts |
| WIF with AWS IAM | No (uses IAM roles) |
| WIF with GCP | No (uses GCP service accounts) |

### WIF with Entra ID architecture:

```
Entra ID (Your Tenant)
 ├── App Registration          ← defines the identity
 └── Service Principal         ← instance in your tenant
          │
          ▼ (federated trust)
     Snowflake Service User    ← trusts the Entra ID identity via WIF
```

The App Registration exists because **Azure issued the identity** — Snowflake just trusts it.

---

## Key Takeaway

- **Databricks** can create its own standalone service principal object (no external dependency).
- **Snowflake** creates a **service user** and optionally federates it to an external identity provider (Entra ID, AWS IAM, or GCP).
- If you use key-pair auth or programmatic access tokens, no external identity provider is involved at all — it's purely within Snowflake.

# Service Principal in Databricks

---

## Two Ways to Get a Service Principal in Databricks

### Case 1: Create in Azure Entra ID first, then use in Databricks

You create a service principal in Entra ID and link it to Databricks:

```
Entra ID (Azure AD)
 ├── App Registration        ← created
 └── Service Principal       ← created
          │
          ▼ (linked to)
     Azure Databricks        ← consumes the Entra ID identity
```

Databricks simply uses the existing Entra ID service principal. No new identity is created inside Databricks — it references the Entra ID one.

**Use this when:** your Databricks workloads need to access Azure resources (ADLS Gen2, Key Vault, Event Hubs, Azure SQL, etc.), because Azure authorization is based on Entra ID identities.

---

### Case 2: Create directly through Databricks Account Console/API

Databricks has its **own identity system**. You can create a service principal purely inside Databricks:

```
Databricks
 └── Service Principal       ← created (Databricks-native)

Entra ID
 └── (nothing created here)
```

**Does this create an App Registration?** **No.** A Databricks service principal is a Databricks-native identity. It does **not** create an App Registration or any object in Entra ID.

**Use this for:**
- SCIM provisioning
- Databricks workspace access
- Account-level administration
- Databricks API authentication (using Databricks OAuth or PATs)

---

## Does Databricks Even Have "App Registrations"?

**No.** The concept of "App Registration" is an **Azure Entra ID construct only**. Databricks does not have its own equivalent of App Registrations.

| Platform | Has App Registration? | Has Service Principal? |
|----------|----------------------|------------------------|
| **Azure Entra ID** | Yes | Yes (linked to App Registration) |
| **Databricks** | No | Yes (standalone, Databricks-native) |

When you create a service principal via Databricks Console:
- Databricks generates a **Databricks-managed identity** with a UUID
- You authenticate using **Databricks OAuth tokens** or **PATs** — not Entra ID client secrets
- No App Registration exists anywhere for this identity

---

## Summary Comparison

| Method | App Registration in Entra ID? | Service Principal in Entra ID? | Service Principal in Databricks? |
|--------|-------------------------------|-------------------------------|----------------------------------|
| `az ad sp create-for-rbac` then link to Databricks | Yes | Yes | Yes (references Entra ID SP) |
| Databricks Account Console / API | No | No | Yes (Databricks-native only) |

---

## When Do You Need Which?

| Scenario | Which to use |
|----------|--------------|
| Access Azure resources (ADLS, Key Vault, etc.) | Entra ID SP (Case 1) |
| Databricks API automation only | Databricks SP (Case 2) |
| Unity Catalog with Azure storage | Entra ID SP (Case 1) |
| Workspace-level automation (jobs, repos) | Either works, but Case 2 is simpler |

**Example — accessing Azure Storage from Databricks:**

```
Databricks Job
   │
   ▼ (uses)
Entra ID Service Principal
   │
   ▼ (authenticates via OAuth)
Azure Storage Account (ADLS Gen2)
```

Here, an App Registration must exist in Entra ID because Azure Storage only recognizes Entra ID identities.

---

## Key Takeaway

- **Databricks service principals ≠ Azure service principals.** They are separate identity constructs.
- Creating a SP in Databricks does **not** touch Entra ID at all.
- Only when you need Azure resource access do you need an Entra ID SP (with its backing App Registration).

# Databricks External Location — How It Differs from Snowflake

Databricks does **not** use the same consent-based multi-tenant pattern as Snowflake. Instead, Databricks offers two approaches — both keep the identity in **your** tenant.

---

## Option 1: Access Connector with Managed Identity (Recommended)

This is the default approach for Unity Catalog:

```
Databricks                               YOUR Azure Tenant
┌─────────────────────────┐             ┌──────────────────────────────┐
│                         │             │                              │
│  Storage Credential     │             │  Access Connector            │
│  (references the        │─────────────►  (Azure-managed resource     │
│   access connector ID)  │             │   with a Managed Identity)   │
│                         │             │                              │
│  External Location      │             │  You assign IAM roles to     │
│  (URL + credential)     │             │  the managed identity        │
└─────────────────────────┘             └──────────────────────────────┘
```

### How it works:

- The **Access Connector** is an Azure resource deployed in **your** subscription/tenant
- It has a **system-assigned managed identity** — Azure manages credentials entirely (no secrets, no rotation needed)
- You grant that managed identity **"Storage Blob Data Contributor"** on your storage
- **No consent URL**, no multi-tenant app, no client secrets involved

---

## Option 2: Service Principal (Manual)

You create and manage your own service principal:

```
Databricks                               YOUR Azure Tenant
┌─────────────────────────┐             ┌──────────────────────────────┐
│                         │             │                              │
│  Storage Credential     │             │  App Registration            │
│  (stores client_id,     │─────────────►  (YOU created this)          │
│   client_secret,        │             │                              │
│   tenant_id)            │             │  Service Principal           │
│                         │             │  (YOU assign IAM roles)      │
└─────────────────────────┘             └──────────────────────────────┘
```

### How it works:

- You create the app registration and service principal **yourself**
- You provide `client_id`, `client_secret`, and `tenant_id` to Databricks
- Databricks stores those credentials and uses them to authenticate
- **You** are responsible for rotating secrets when they expire

---

## Comparison: Snowflake vs Databricks

| Aspect | Snowflake | Databricks (Access Connector) | Databricks (Manual SP) |
|--------|-----------|-------------------------------|------------------------|
| **Who creates the identity?** | Snowflake (in their tenant) | Azure (managed identity in your tenant) | You (in your tenant) |
| **Where does the identity live?** | Snowflake's tenant → consented into yours | Your tenant (your Azure resource) | Your tenant |
| **Credentials managed by?** | Snowflake (you never see them) | Azure (fully managed, no secrets) | You (must rotate secrets) |
| **Consent URL needed?** | Yes | No | No |
| **Multi-tenant app?** | Yes | No | No |
| **Your control?** | IAM role assignment only | Full (it's your resource) | Full |

---

## The Core Difference in Philosophy

| Platform | Pattern | Summary |
|----------|---------|--------|
| **Snowflake** | Multi-tenant OAuth | "We own the identity, you consent to let us in" |
| **Databricks (recommended)** | Managed Identity | "You own the identity, it lives in your subscription, Azure manages it — zero secrets" |
| **Databricks (manual)** | Traditional SP | "You own and manage everything yourself — you handle rotation" |

Databricks' recommended approach keeps everything in your own tenant with **zero shared secrets** and no external party holding credentials to your resources.